# Tái hiện benchmark DTIN của MagicLens

Notebook này chạy lại kết quả **Domain-Transfer ImageNet** (R@10, R@50 cho 4 miền cartoon, origami, toy, sculpture) bằng mã gốc trong repo [google-deepmind/magiclens](https://github.com/google-deepmind/magiclens), rồi so với Bảng 15 của bài báo.

**Benchmark này đo gì.** Có 10 000 truy vấn là ảnh chụp thật (tập val của ImageNet) kèm chỉ dẫn `"find this object in {miền}"`. Kho tìm kiếm có 16 983 ảnh: 6 983 ảnh ImageNet-R của 4 miền, cộng với chính 10 000 ảnh chụp thật đóng vai ảnh gây nhiễu. Một truy vấn được tính là đúng nếu top-K có ảnh cùng lớp trong đúng miền được yêu cầu.

## Trước khi chạy

**1. Chọn GPU.** Vào **Runtime → Change runtime type**, chọn **A100 GPU**, bấm **Save**.

**2. Tập val của ImageNet.** Notebook cần tệp `ILSVRC2012_img_val.tar` (6.4 GB). Nếu Drive chưa có tệp này, ô 8 thử tải thẳng từ image-net.org, giống lệnh trong `data/README.md` của repo. Nếu không tải được, hãy đăng ký tài khoản trên [image-net.org](https://image-net.org) (miễn phí cho mục đích nghiên cứu), tải tệp về, rồi đặt vào `MyDrive/magiclens/data/dtin/` trên Drive. Không cần giải nén. Notebook chỉ lấy đúng 10 000 ảnh cần dùng.

Các phần còn lại (danh sách truy vấn, ảnh ImageNet-R) được tải tự động. Nếu máy chủ ImageNet-R không truy cập được, có thể tự tải `imagenet-r.tar` rồi đặt vào cùng thư mục trên Drive.

## Cách chạy

- Chạy lần lượt từng ô từ trên xuống: bấm vào ô rồi nhấn **Shift + Enter**, hoặc bấm nút ▶ bên trái ô. Muốn chạy hết một lượt thì chọn **Runtime → Run all**.
- Ô 4 sẽ hỏi quyền truy cập Google Drive: chọn tài khoản của bạn và đồng ý.
- Kết quả (bảng so sánh, log, tệp top-50) được lưu vào `MyDrive/magiclens/ket_qua/dtin/` trên Google Drive.
- Mỗi lần mở lại notebook là một máy Colab mới, nên phải chạy lại từ ô 1. Trong cùng một phiên, ô nào đã làm xong sẽ tự bỏ qua.
- Pip có thể in cảnh báo bắt đầu bằng `ERROR: pip's dependency resolver...`. Đây chỉ là cảnh báo xung đột phiên bản với các gói có sẵn của Colab, **bỏ qua được**. Notebook chỉ dừng khi có lỗi thật.
- Xong việc thì vào **Runtime → Disconnect and delete runtime** để không tốn compute unit.

## Bước 1: Cấu hình

In [ ]:
# Ô 1: Cấu hình. Chỉ cần sửa ở ô này.
MODEL_SIZE = "base"          # "base" hoặc "large"
BATCH_SIZE = None            # None = tự chọn; giảm (ví dụ 256) nếu báo RESOURCE_EXHAUSTED
MIEN = ["cartoon", "origami", "toy", "sculpture"]

DRIVE_DIR = "/content/drive/MyDrive/magiclens"                       # thư mục làm việc trên Google Drive
IMAGENET_VAL_TAR = DRIVE_DIR + "/data/dtin/ILSVRC2012_img_val.tar"   # nếu không có, notebook thử tải từ image-net.org
IMAGENET_R_TAR = DRIVE_DIR + "/data/dtin/imagenet-r.tar"             # không bắt buộc; nếu không có sẽ tự tải
REPO = "/content/magiclens"                                          # nơi đặt mã nguồn trên máy Colab
BENCHMARK = "dtin"

## Bước 2: Chuẩn bị máy Colab

Các ô 2–7 kiểm tra máy, kết nối Drive, cài thư viện, tải checkpoint và chạy thử mô hình. Ở ô 7, dòng cuối phải có `chạy trên: GPU`. Nếu là `CPU` thì runtime chưa có GPU hoặc JAX chưa nhận GPU: kiểm tra lại ô 3 và ô 5.

In [ ]:
# Ô 2: Các hàm dùng chung. Chạy ô này, không cần sửa gì.
import ast, codecs, glob, json, os, re, shlex, shutil, subprocess, sys, tarfile, time, zipfile

try:
    import google.colab  # có mặt khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"           # bớt log của TensorFlow
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"   # TensorFlow không giữ trước bộ nhớ GPU
PY = sys.executable                                 # trình Python dùng để cài thư viện và chạy mã
CONTENT = os.path.dirname(REPO)                     # /content trên Colab
LOG_DIR = os.path.join(CONTENT, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
q = shlex.quote


def sh(cmd, log=None, check=True):
    # Chạy một lệnh shell, in kết quả ngay khi có và ghi vào tệp log nếu cần.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")(errors="replace")
    f = open(log, "wb") if log else None
    try:
        while True:
            chunk = p.stdout.read1(65536)
            if not chunk:
                break
            sys.stdout.write(dec.decode(chunk))
            sys.stdout.flush()
            if f:
                f.write(chunk)
    finally:
        if f:
            f.close()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh bị lỗi (mã {rc}). Đọc thông báo ngay phía trên để biết nguyên nhân.")
    return rc


def chay_python(ma, cwd=None):
    # Ghi đoạn mã ra tệp rồi chạy bằng một tiến trình Python riêng.
    thu_muc = cwd or CONTENT
    tep = os.path.join(thu_muc, "_chay_thu.py")
    with open(tep, "w", encoding="utf-8") as f:
        f.write(ma)
    return sh(f"cd {q(thu_muc)} && {q(PY)} {q(tep)}")


def ket_qua_dir():
    d = os.path.join(DRIVE_DIR, "ket_qua", BENCHMARK, MODEL_SIZE)
    os.makedirs(d, exist_ok=True)
    return d


def chay_inference(dataset, ten_log, env="", output="./output"):
    # Chạy inference.py của repo cho một bộ dữ liệu, ghi log, trả về đường dẫn log.
    bs = BATCH_SIZE or (1024 if MODEL_SIZE == "base" else 512)
    log = os.path.join(LOG_DIR, ten_log)
    cmd = (f"cd {q(REPO)} && {env} {q(PY)} inference.py --model_size {MODEL_SIZE} "
           f"--model_path ./models/magic_lens_clip_{MODEL_SIZE}.pkl --dataset {dataset} "
           f"--batch_size {bs} --output {output}")
    t0 = time.time()
    sh(cmd, log=log)
    print(f"\nXong {dataset} sau {(time.time() - t0) / 60:.1f} phút. Log: {log}")
    shutil.copy(log, ket_qua_dir())
    return log


def doc_recall(log):
    # Đọc dòng "Recalls:  {10: ..., 50: ...}" mà inference.py in ra.
    text = open(log, encoding="utf-8", errors="replace").read()
    found = re.findall(r"Recalls:\s*(\{[^}]*\})", text)
    if not found:
        raise RuntimeError(f"Không thấy dòng 'Recalls:' trong {log}.")
    r = ast.literal_eval(found[-1])
    return r[10], r[50]


def bang(tieu_de, cac_dong):
    # Tạo bảng Markdown.
    dong = ["| " + " | ".join(tieu_de) + " |", "|" + "---|" * len(tieu_de)]
    dong += ["| " + " | ".join(str(x) for x in d) + " |" for d in cac_dong]
    return "\n".join(dong)


def hien_thi(md):
    try:
        from IPython.display import Markdown, display
        display(Markdown(md))
    except ImportError:
        print(md)


def con_trong_gb(path):
    return shutil.disk_usage(path).free / 1e9


print("Sẵn sàng. Đang chạy trên Colab:", IN_COLAB)

In [ ]:
# Ô 3: Kiểm tra máy Colab: GPU, RAM, ổ đĩa
if sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False) != 0:
    print("\nCẢNH BÁO: không thấy GPU. Vào Runtime → Change runtime type → chọn A100 GPU → Save, rồi chạy lại từ ô 1.")
sh("free -g | head -2", check=False)
print(f"Ổ đĩa còn trống: {con_trong_gb(CONTENT):.0f} GB")
print("DTIN cần khoảng 16 GB RAM, vì mã giữ toàn bộ 26 983 ảnh (truy vấn và kho) trong RAM.")

In [ ]:
# Ô 4: Kết nối Google Drive. Colab sẽ hỏi quyền truy cập: chọn tài khoản của bạn và đồng ý.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Thư mục làm việc trên Drive:", DRIVE_DIR)

In [ ]:
# Ô 5: Tải mã nguồn và cài thư viện. Lần đầu mất vài phút; chạy lại trong cùng phiên sẽ tự bỏ qua.
DA_CAI = os.path.join(CONTENT, ".magiclens_da_cai")
if os.path.exists(DA_CAI):
    print("Đã cài trong phiên này, bỏ qua.")
else:
    if not os.path.isdir(REPO):
        sh(f"git clone -q https://github.com/google-deepmind/magiclens.git {q(REPO)}")
    scenic = os.path.join(CONTENT, "scenic")
    if not os.path.isdir(scenic):
        sh(f"git clone -q https://github.com/google-research/scenic.git {q(scenic)}")
    pip = f"{q(PY)} -m pip install -q"
    sh(f"{pip} {q(scenic)}")
    sh(f"{pip} -r {q(scenic + '/scenic/projects/baselines/clip/requirements.txt')}")
    # Bộ phiên bản đã chạy được với mã này. Nếu không cài được thì dùng bản mới nhất.
    if sh(f'{pip} "jax[cuda12]==0.11.1" "flax==0.12.9"', check=False) != 0:
        print("Không cài được bộ phiên bản đã kiểm thử, chuyển sang bản mới nhất.")
        sh(f'{pip} -U "jax[cuda12]" flax')
    open(DA_CAI, "w").close()
chay_python("import jax, flax\nprint('jax', jax.__version__, '| flax', flax.__version__, '|', jax.devices())\n")

In [ ]:
# Ô 6: Tải checkpoint MagicLens (công khai, không cần đăng nhập). Bản base 666 MB, bản large 1.9 GB.
KICH_THUOC = {"base": 665747853, "large": 1861692696}
CKPT = os.path.join(REPO, "models", f"magic_lens_clip_{MODEL_SIZE}.pkl")
if os.path.exists(CKPT) and os.path.getsize(CKPT) == KICH_THUOC[MODEL_SIZE]:
    print("Đã có checkpoint:", CKPT)
else:
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    url = f"https://storage.googleapis.com/gresearch/magiclens/models/magic_lens_clip_{MODEL_SIZE}.pkl"
    sh(f"wget -q --show-progress -c -O {q(CKPT)} {url}")
    if os.path.getsize(CKPT) != KICH_THUOC[MODEL_SIZE]:
        raise RuntimeError("Tệp tải về chưa đủ dung lượng. Chạy lại ô này để tải tiếp.")
    print("Đã tải:", CKPT)

In [ ]:
# Ô 7: Chạy thử mô hình một lần để chắc mọi thứ đã sẵn sàng
chay_python(
    "import jax, jax.numpy as jnp\n"
    "from inference import load_model\n"
    f"model, params = load_model('{MODEL_SIZE}', 'models/magic_lens_clip_{MODEL_SIZE}.pkl')\n"
    "out = model.apply(params, {'ids': jnp.ones((1, 1, 77), jnp.int32), 'image': jnp.ones((1, 224, 224, 3))})\n"
    "print('Embedding:', out['multimodal_embed_norm'].shape, '| chạy trên:', jax.devices()[0].platform.upper())\n",
    cwd=REPO)

## Bước 3: Dữ liệu

Ô 8 tải hai danh sách ảnh của Pic2Word, tải và giải nén ImageNet-R, rồi lấy đúng 10 000 ảnh chụp thật từ tệp tar val (trên Drive, hoặc tải thẳng nếu Drive chưa có). Ảnh được xếp vào `data/dtin/imagenet-r/` và `data/dtin/real/` đúng như mã yêu cầu.

In [ ]:
# Ô 8: Chuẩn bị dữ liệu DTIN
DTIN_DIR = os.path.join(REPO, "data", "dtin")
os.makedirs(DTIN_DIR, exist_ok=True)
URL_DS = "https://raw.githubusercontent.com/google-research/composed_image_retrieval/refs/heads/main/data/imgnet"
for ten in ["imgnet_real_query.txt", "imgnet_targets.txt"]:
    if not os.path.exists(os.path.join(DTIN_DIR, ten)):
        sh(f"wget -q -O {q(os.path.join(DTIN_DIR, ten))} {URL_DS}/{ten}")


def anh_can_co():
    can = set()
    for ten in ["imgnet_real_query.txt", "imgnet_targets.txt"]:
        for dong in open(os.path.join(DTIN_DIR, ten)):
            if dong.strip():
                can.add(dong.split()[0].split("/", 1)[1])   # bỏ tiền tố "imgnet/", giống data_utils.py
    return sorted(can)


def con_thieu():
    return [p for p in anh_can_co() if not os.path.exists(os.path.join(DTIN_DIR, p))]


# 1) Ảnh ImageNet-R của 4 miền
if any(p.startswith("imagenet-r/") for p in con_thieu()):
    tar_r, tu_tai = IMAGENET_R_TAR, False
    if not os.path.exists(tar_r):
        tar_r, tu_tai = os.path.join(CONTENT, "imagenet-r.tar"), True
        sh(f"wget -q --show-progress -c -O {q(tar_r)} https://people.eecs.berkeley.edu/~hendrycks/imagenet-r.tar")
    sh(f"tar -xf {q(tar_r)} -C {q(DTIN_DIR)}")
    if tu_tai:
        os.remove(tar_r)

# 2) Ảnh chụp thật: chỉ lấy đúng các ảnh cần từ tệp tar val của ImageNet
thieu_real = {os.path.basename(p): p for p in con_thieu() if p.startswith("real/")}
if thieu_real:
    tar_val = IMAGENET_VAL_TAR
    if not os.path.exists(tar_val):
        # Drive chưa có: thử tải thẳng như data/README.md của repo
        tar_val = os.path.join(CONTENT, "ILSVRC2012_img_val.tar")
        print(f"Không thấy {IMAGENET_VAL_TAR}: thử tải thẳng từ image-net.org (6.4 GB).", flush=True)
        rc = sh(f"wget -q --show-progress -c -O {q(tar_val)} "
                "https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar", check=False)
        if rc != 0 or os.path.getsize(tar_val) < 6_000_000_000:
            if os.path.exists(tar_val) and os.path.getsize(tar_val) < 100_000_000:
                os.remove(tar_val)   # trang báo lỗi, không phải tệp tar
            raise FileNotFoundError(
                "Không tải được ILSVRC2012_img_val.tar từ image-net.org. Đăng nhập image-net.org, tải tệp này "
                f"về máy rồi đưa lên Drive, đặt đúng tại {IMAGENET_VAL_TAR} (xem mục 'Trước khi chạy').")
    print(f"Lấy {len(thieu_real)} ảnh từ {tar_val} ...", flush=True)
    with tarfile.open(tar_val) as tf:
        for m in tf:
            ten = os.path.basename(m.name)
            if m.isfile() and ten in thieu_real:
                dich = os.path.join(DTIN_DIR, thieu_real.pop(ten))
                os.makedirs(os.path.dirname(dich), exist_ok=True)
                with tf.extractfile(m) as nguon, open(dich, "wb") as ra:
                    shutil.copyfileobj(nguon, ra)
            if not thieu_real:
                break

con = con_thieu()
print(f"Cần {len(anh_can_co())} ảnh (bản đầy đủ: 16983), còn thiếu {len(con)}.")
if con:
    raise RuntimeError("Còn thiếu ảnh, ví dụ: " + ", ".join(con[:3]))

## Bước 4: Chạy đánh giá

Mỗi miền chạy `inference.py` một lần. Kho ảnh giống hệt nhau ở cả 4 miền, nhưng mã gốc vẫn nạp và mã hóa lại toàn bộ ở mỗi lần chạy.

In [ ]:
# Ô 9: Chạy đánh giá cho 4 miền
KET_QUA = {}
for d in MIEN:
    log = chay_inference(f"dtin-{d}", f"dtin-{d}_{MODEL_SIZE}.log")
    KET_QUA[d] = doc_recall(log)
    top50 = os.path.join(REPO, "output", f"dtin-{d}_{MODEL_SIZE}", f"dtin-{d}_results.json")
    if os.path.exists(top50):
        shutil.copy(top50, ket_qua_dir())
    print(f"==> {d}: R@10 = {KET_QUA[d][0]:.2f}, R@50 = {KET_QUA[d][1]:.2f}\n")

## Bước 5: So sánh với bài báo

In [ ]:
# Ô 10: So sánh với Bảng 15 của bài báo và lưu lên Drive
BAI_BAO = {
    "base":  {"cartoon": (49.4, 67.0), "origami": (13.8, 26.3), "toy": (25.8, 43.4), "sculpture": (24.3, 41.3), "trung bình": (28.3, 44.5)},
    "large": {"cartoon": (62.6, 72.2), "origami": (21.5, 33.4), "toy": (43.8, 58.4), "sculpture": (38.0, 54.2), "trung bình": (41.5, 54.5)},
}[MODEL_SIZE]


def dong_ket_qua(ten, r10, r50):
    b10, b50 = BAI_BAO[ten]
    return [ten, f"{r10:.1f} / {r50:.1f}", f"{b10} / {b50}", f"{r10 - b10:+.1f} / {r50 - b50:+.1f}"]


cac_dong = [dong_ket_qua(d, *KET_QUA[d]) for d in KET_QUA]
if len(KET_QUA) == 4:
    cac_dong.append(dong_ket_qua("trung bình", sum(v[0] for v in KET_QUA.values()) / 4,
                                 sum(v[1] for v in KET_QUA.values()) / 4))
md_kq = (f"### DTIN, MagicLens-{MODEL_SIZE} (R@10 / R@50)\n\n"
         + bang(["Miền", "Tái hiện", "Bài báo", "Chênh lệch"], cac_dong))
hien_thi(md_kq)
with open(os.path.join(ket_qua_dir(), "tong_hop.md"), "w", encoding="utf-8") as f:
    f.write(md_kq + "\n")
with open(os.path.join(ket_qua_dir(), "ket_qua.json"), "w") as f:
    json.dump({d: {"R@10": v[0], "R@50": v[1]} for d, v in KET_QUA.items()}, f, indent=1)
print("Đã lưu vào", ket_qua_dir())
print("Checkpoint công bố là bản chuyển đổi, nên lệch dưới khoảng 1 điểm là bình thường.")

## Chạy bản Large

Ở ô 1, đổi `MODEL_SIZE = "large"`, chạy lại ô 1, rồi chạy tiếp từ ô 6 trở xuống. Dữ liệu đã chuẩn bị trong phiên này được dùng lại. 

## Kết thúc

Kết quả đã nằm trên Drive. Để không tốn compute unit, vào **Runtime → Disconnect and delete runtime**, hoặc đổi `NGAT_KET_NOI = True` ở ô dưới rồi chạy ô đó.

In [ ]:
# Ô cuối: Ngắt kết nối máy Colab. Mọi thứ trên máy sẽ bị xóa, kết quả trên Drive vẫn còn.
NGAT_KET_NOI = False   # đổi thành True rồi chạy ô này khi đã xong
if NGAT_KET_NOI and IN_COLAB:
    from google.colab import runtime
    runtime.unassign()